# Pipeline for setting up a toy dataset for training models

Here is a brief description of how to load the toy dataset, as well as a brief description of its components.

It is important to have the [Partitura](https://github.com/CPJKU/partitura) package set up in your environment (`pip install partitura`).
The note features (`W`) are the ones of `partitura.musicanalysis.note_features` (formerly called basis functions in the Basis Mixer).

In [ ]:
import glob
import os

import numpy as np
import partitura

from partitura.musicanalysis.performance_codec import (
    get_unique_onset_idxs,
    onsetwise_to_notewise,
    notewise_to_onsetwise,
)

Dowload the files (or alternatively, set up the path to the data), and load the files. Each `.npz` file contains the information for a single piece.

In [ ]:
# Download dataset or set a path to the dataset
download_dataset = True

if download_dataset:
    import urllib.request
    import zipfile
    url = 'https://jobim.ofai.at/downloads/carlos/synthetic_performance/synthetic_expressive_dataset.zip'
    urllib.request.urlretrieve(url, 'synthetic_expressive_dataset.zip')
    # set a directory for storing the dataset
    download_dir = '/tmp/dataset'
    if not os.path.exists(data_dir):
        os.mkdir(data_dir)
    with zipfile.ZipFile('synthetic_expressive_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall(download_dir)
    data_dir = os.path.join(download_dir, 'npz')
else:
    # set a path for the dataset
    data_dir = 'path/to/dataset'

filenames = glob.glob(os.path.join(data_dir, '*.npz'))
# Load dataset
dataset = []
piecenames = []
for i, fn in enumerate(filenames):
    dataset.append(dict(np.load(fn).items()))
    piecenames.append(os.path.basename(fn).replace('.npz', ''))
    
print('There ar {0} pieces'.format(len(filenames)))

Each of the files contains the following data

* `score_info`: A structured array which contains score information. Each row in the array represents information about a note in the score. This array contains the following keys:
    * `onset`: Onset time of the note in beats (i.e., where does the note start in score time)
    * `duration`: Symbolic duration of the note (in beats)
    * `pitch`: MIDI pitch of the note
    * `id`: a string representing the ID of the note in the original score file (e.g., for MusicXML files would be the property `id` for `note` tags, for example `<note id="n01">`).
* `W`: A 2D numpy array containing musical information extracted from the score using basis functions (i.e., the input data!). Each row in the array represents information for a note and each column represents a basis function.
* `bf_names`. A list of names of the basis functions. The i-th element of this list corresponds to the i-th column of `W`.
* `predictions`: A structured array which contains performance information encoded in expressive parameters. Each row in the array represents the performance information of a note in the score (the i-th row in this array corresponds to the i-th row in `score_info`). This array has the following keys:
    * `beat_period`: Beat period in seconds per beat (i.e., $bp = \frac{60}{\text{beats per minute}}$) represents the expressive tempo. The beat period represents the tempo at the current score position (i.e., all notes that start at the same score onset have the same beat period). A larger value of beat period means a slower performance.
    * `velocity_trend`: The MIDI velocity is decomposed into two parameters: a trend for each score position (all of the notes starting at the same score onset have the same value) and 
    * `velocity_deviation`: The deviation in MIDI velocity from the trend. The MIDI velocitiy of a note is computed as
    $$\text{velocity}(\text{note}_i) = \text{velocity}_{\text{trend}}(\text{note}_i) - \text{velocity}_{\text{deviations}}(\text{note}_i)$$
    * `timing`: Deviations of the onsets of each note (in seconds) from the temporal position implied by `beat_period`.
    * `articulation_log`: Logarithm of the articulation ratio, the ratio between the performed duration and the notated duration at the current tempo, i.e.
    $$\text{articulation}_{log} = \log_2 \frac{\text{performed duration}}{\text{notated duration}\times\text{beat period}}$$
    
Since the basis functions used for each piece depend on the information on the score (e.g., the time signature, dynamics markings, etc). For example, if there are is no $\boldsymbol{p}$ marking in the piece, there will not be a column in `W` representing $\boldsymbol{p}$. This means that not all pieces generate the same number of basis functions (and thus, the shape of `W` is different for each piece). 

To create the input data we have to get all basis functions used for all pieces

In [ ]:
# Name of basis functions
bf_names = np.array(list(set([b for bl in [list(d['bf_names']) for d in dataset] for b in bl])))
bf_names.sort()
# print(bf_names)
n_bfs = len(bf_names)
target_names = np.array(dataset[0]['predictions'].dtype.names)

Finally, we get the inputs and targets for each piece:

* `X_notewise` and `Y_notewise` are lists containing the array of input features and targets for each piece for each note, respectively. Each element in each of these lists is an array whose i-th row are the input features/targets for each note.
* `X_onsetwise` and `Y_notewise` are lists containing the array of input features and targets for each piece aggregated for each onset. Each element in these lists is an array whose i-th row are the input features/targets for each score onset (see `u_onsets` and `u_onset_idxs`)
* `u_onsets` and `u_onset_idxs` are lists containing the unique score onsets for each piece and the indices of the arrays in the notewise arrays corresponding to these onsets.

In [ ]:
# List of Inputs Features (notewise and onsetwise) (for each piece)
X_notewise, X_onsetwise = [], []
# Targets for each piece
Y_notewise, Y_onsetwise = [], []
# Unique score onsets and score onset indices (the index of each note in the same score onset)
u_onsets, u_onset_idxs = [], []
# Get inputs and features
for piece in dataset:
    X_piece = np.zeros((len(piece['W']), n_bfs))
    # Indices of columns corresponding to the basis functions in the full list
    bf_idxs = np.array([int(np.where(bf_names == bfn)[0]) for bfn in piece['bf_names']], dtype=int)
    X_piece[:, bf_idxs] = piece['W']
    onsets = piece['score_info']['onset']
    # get unique onsets and unique onset indices
    u_on_idxs, u_on = get_unique_onset_idxs(onsets, return_unique_onsets=True)
    u_onsets.append(u_on)
    u_onset_idxs.append(u_on_idxs)
    X_notewise.append(X_piece)
    Y_notewise.append(np.array(piece['predictions'].tolist()))
    # Aggregate the notewise arrays to onsetwise
    X_onsetwise.append(notewise_to_onsetwise(X_piece, u_on_idxs))
    Y_onsetwise.append(notewise_to_onsetwise(Y_notewise[-1], u_on_idxs))
    